# Post-Milo analysis
This notebook demonstrates basic downstream analysis of Milo differential abundance results.

In [ ]:
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

## Load Milo result datasets

In [ ]:
outdir = 'results'  # directory used in custom_design_pipeline
Designs = ['acr','cr','pc','pac']
adata_dict = {d: sc.read_h5ad(os.path.join(outdir, f'custom_design.{d}.post_milo.h5ad')) for d in Designs}

## Inspect top neighborhoods by significance

In [ ]:
for name, ad in adata_dict.items():
    df = ad.uns['nhood_adata'].obs
    top = df.sort_values('SpatialFDR').head()
    display(name, top[['nhood_annotation','logFC','SpatialFDR']])

## Count significant neighborhoods per cell type

In [ ]:
results=[]
for name, ad in adata_dict.items():
    df = ad.uns['nhood_adata'].obs
    sig = df[df['SpatialFDR']<0.05]
    counts = sig['nhood_annotation'].value_counts().rename('count').reset_index().rename(columns={'index':'cell_type'})
    counts['design']=name
    results.append(counts)
count_df = pd.concat(results)
count_df

In [ ]:
pivot = count_df.pivot_table(index='cell_type', columns='design', values='count', fill_value=0)
pivot.plot(kind='bar', figsize=(8,4))
plt.ylabel('Significant neighborhood count')
plt.tight_layout()
plt.show()